# Chapter 2 — Convolution Fundamentals: Kernels, Feature Maps, Receptive Field

## Learning Objectives
- Understand the convolution operation mathematically and visually
- Implement 2D convolution from scratch in NumPy
- Understand kernels, strides, padding, and their effect on output size
- Visualise what different kernels detect in satellite images
- Understand receptive field and why it grows with depth
- Use PyTorch's nn.Conv2d and understand all its parameters

## Estimated Duration
Theory: 3h | Practical: 2h | Total: 5h

Difficulty: Beginner

## Key Concepts
- Discrete 2D convolution vs cross-correlation
- Kernel / filter / weight matrix
- Feature map / activation map
- Padding: 'valid' vs 'same'
- Stride: downsampling via convolution
- Receptive field: how many pixels does a neuron 'see'?

In [1]:
# ─── COLAB SETUP (run this cell first) ────────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q torch torchvision torchgeo matplotlib seaborn

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = Path('./data') if not IN_COLAB else Path('/content/data')
print(f'Device: {DEVICE}')

Device: cpu


/home/francesco/personal/ML_courses/CNN_in_EO/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 2.1 — The Convolution Operation: Intuition

A convolution slides a small filter (kernel) across an image and computes
a dot product at each position. The result is a feature map.

For a kernel $K$ of size $k \times k$ and input image $I$ of size $H \times W$:

$$\text{output}[i,\, j] = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} K[m,\, n] \cdot I[i+m,\, j+n]$$

Key insight: The SAME kernel is applied at EVERY position.
This is called **weight sharing** and is what makes CNNs efficient.

> **Note:** PyTorch and most DL frameworks implement cross-correlation
> (no kernel flip), calling it "convolution" by convention.

In [ ]:
# ─── NumPy convolution from scratch ─────────────────────────────────────────

def convolve2d_numpy(image: np.ndarray, kernel: np.ndarray, padding: int = 0) -> np.ndarray:
    """
    2D cross-correlation (what DL frameworks call 'convolution').
    
    Parameters
    ----------
    image   : (H, W) float array
    kernel  : (kH, kW) float array
    padding : number of zero-padding pixels on each side
    
    Returns
    -------
    output : (H', W') float array
    """
    H, W = image.shape
    kH, kW = kernel.shape
    
    # Pad the image with zeros
    if padding > 0:
        image = np.pad(image, padding, mode='constant', constant_values=0)
    
    H_pad, W_pad = image.shape
    out_H = H_pad - kH + 1
    out_W = W_pad - kW + 1
    
    output = np.zeros((out_H, out_W), dtype=np.float32)
    
    for i in range(out_H):
        for j in range(out_W):
            # Extract the patch
            patch = image[i:i+kH, j:j+kW]
            # Dot product with kernel
            output[i, j] = np.sum(patch * kernel)
    
    return output


# Test on a simple 5×5 example
test_image = np.array([
    [1, 2, 3, 0, 1],
    [4, 5, 6, 1, 2],
    [7, 8, 9, 2, 3],
    [1, 2, 3, 4, 5],
    [2, 3, 4, 5, 6],
], dtype=np.float32)

# Sobel kernel: detects horizontal edges
sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=np.float32)

result = convolve2d_numpy(test_image, sobel_x, padding=0)
print('Input image:')
print(test_image)
print('\nSobel-X kernel:')
print(sobel_x)
print('\nConvolution result (edge response):')
print(result)
print(f'\nInput shape: {test_image.shape}  →  Output shape: {result.shape}')
print('No padding: output = (H - kH + 1) × (W - kW + 1) = (5-3+1) × (5-3+1) = 3×3')

In [ ]:
# Verify our implementation matches PyTorch
image_t = torch.tensor(test_image).unsqueeze(0).unsqueeze(0)  # (1, 1, 5, 5)
kernel_t = torch.tensor(sobel_x).unsqueeze(0).unsqueeze(0)    # (1, 1, 3, 3)

pytorch_result = nn.functional.conv2d(image_t, kernel_t, padding=0)

print('Our NumPy result:')
print(result)
print('\nPyTorch result:')
print(pytorch_result.squeeze().numpy())
print(f'\nMax absolute difference: {np.abs(result - pytorch_result.squeeze().numpy()).max():.2e}')
print('Perfect match! Our implementation is correct.')

## 2.2 — Classic Image Processing Kernels

Before CNNs, image processing relied on hand-designed kernels.
Understanding these gives intuition for what CNN filters learn:

Edge detectors (Sobel, Prewitt, Laplacian)  →  CNN early layers learn similar patterns
Blur filters (Gaussian)                      →  CNN depth = hierarchy of blur + edge responses
Sharpening kernels                           →  High-frequency feature enhancement

In [ ]:
# Load a real satellite image from EuroSAT for demonstrations
from torchgeo.datasets import EuroSAT

EUROSAT_ROOT = DATA_ROOT / "eurosat"
EUROSAT_ROOT.mkdir(parents=True, exist_ok=True)
dataset = EuroSAT(root=EUROSAT_ROOT, split="train", download=True)

# Find a Residential sample (high texture, good for edge detection)
for sample in dataset:
    if dataset.classes[sample["label"].item()] == "Residential":
        residential_img = sample["image"].float()
        break

# Convert to grayscale for single-channel kernel demo
# Use Red channel (index 2 for Sentinel-2 RGB ordering in torchgeo)
gray = residential_img[2].numpy()  # Red band
# Normalize for display
gray = (gray - gray.min()) / (gray.max() - gray.min() + 1e-8)

In [ ]:
# Define classic kernels
kernels = {
    'Sobel X\n(vertical edges)': np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32),
    'Sobel Y\n(horizontal edges)': np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=np.float32),
    'Laplacian\n(all edges)': np.array([[0,-1,0],[-1,4,-1],[0,-1,0]], dtype=np.float32),
    'Gaussian blur\n(3×3 ÷16)': np.array([[1,2,1],[2,4,2],[1,2,1]], dtype=np.float32) / 16,
    'Sharpen': np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32),
    'Identity': np.array([[0,0,0],[0,1,0],[0,0,0]], dtype=np.float32),
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes[0, 0].imshow(gray, cmap='gray')
axes[0, 0].set_title('Original (Red band, Residential)', fontsize=9)
axes[0, 0].axis('off')

axes[1, 0].imshow(np.zeros_like(gray), cmap='gray')  # placeholder
axes[1, 0].axis('off')

for ax, (name, kernel) in zip(list(axes.ravel())[1:], kernels.items()):
    result = convolve2d_numpy(gray, kernel, padding=1)  # same padding
    # Clip for display
    result_display = np.clip(result, -1, 1)
    ax.imshow(result_display, cmap='RdBu_r' if 'edge' in name.lower() or 'sobel' in name.lower() or 'laplacian' in name.lower() else 'gray')
    ax.set_title(name, fontsize=9)
    ax.axis('off')

plt.suptitle('Hand-Designed Kernels Applied to Sentinel-2 Residential Patch\n'
             '(CNN filters learn variants of these patterns automatically)', fontsize=11)
plt.tight_layout()
plt.show()

## 2.3 — Padding: 'valid' vs 'same'

Without padding (valid): output shrinks by (kernel_size - 1) per dimension

With padding = k//2 (same): output has same spatial dimensions as input

Output size formula:
  out_size = floor((in_size + 2*padding - kernel_size) / stride) + 1

Example for in=64, k=3, stride=1:
  padding=0: out = 62  (shrinks!)
  padding=1: out = 64  (same size)

In practice: use padding=kernel_size//2 for 'same' padding.

In [ ]:
def output_size(in_size: int, kernel_size: int, stride: int = 1, padding: int = 0) -> int:
    """Standard CNN output size formula."""
    return (in_size + 2 * padding - kernel_size) // stride + 1

# Demonstrate the effect of padding and stride
print('Effect of padding and stride on feature map size (input=64×64):')
print(f'{"Configuration":<35} {"Output size":>12}')
print('-' * 50)

configs = [
    ('k=3, stride=1, pad=0 (valid)', 3, 1, 0),
    ('k=3, stride=1, pad=1 (same)',  3, 1, 1),
    ('k=5, stride=1, pad=0',         5, 1, 0),
    ('k=5, stride=1, pad=2 (same)',  5, 1, 2),
    ('k=3, stride=2, pad=0',         3, 2, 0),
    ('k=3, stride=2, pad=1',         3, 2, 1),
    ('k=7, stride=2, pad=3',         7, 2, 3),
]

for name, k, s, p in configs:
    out = output_size(64, k, s, p)
    print(f'{name:<35} {out:>12}×{out}')

print('\nKey rule: padding = kernel_size // 2 → "same" padding (output = input size)')
print('Key rule: stride = 2 → halves spatial dimensions (like MaxPool2d)')

## 2.4 — Receptive Field

The receptive field of a neuron is the spatial region of the input image
that can influence its activation.

For a stack of N conv layers (no pooling, k×k kernels, stride=1):
  
  RF = 1 + N × (k - 1)

For 3×3 kernels:
  - 1 layer:   RF = 3
  - 2 layers:  RF = 5
  - 5 layers:  RF = 9 (AlexNet Layer 1 = 11×11 = 1 layer with 11×11 kernel!)
  - 10 layers: RF = 19 (VGG block)
  - 50 layers: RF = 99

With MaxPool (stride=2): RF effectively doubles for subsequent layers.

This is why DEEPER networks can detect LARGER objects — the receptive
field grows and later layers 'see' more of the scene.

For EO at 10m/pixel:
  RF = 99 pixels → 990m coverage → can detect field boundaries, urban blocks

In [ ]:
# Visualise how receptive field grows with depth

def receptive_field_with_pool(n_conv_per_pool: int, n_pools: int, k: int = 3) -> list:
    """
    Compute cumulative receptive field at each layer in a network
    with alternating conv blocks and MaxPool2d(stride=2).
    """
    rf = 1
    stride_acc = 1
    history = [('Input', rf)]
    
    for pool_idx in range(n_pools):
        for conv_idx in range(n_conv_per_pool):
            rf = rf + stride_acc * (k - 1)
            history.append((f'Conv (pool{pool_idx+1}, conv{conv_idx+1})', rf))
        # After MaxPool: stride doubles
        stride_acc *= 2
        history.append((f'MaxPool {pool_idx+1}', rf))
    
    return history

vgg_like = receptive_field_with_pool(n_conv_per_pool=2, n_pools=4, k=3)

print('Receptive field growth (VGG-like architecture, 3×3 kernels):')
for layer_name, rf in vgg_like:
    bar = '█' * (rf // 2)
    print(f'  {layer_name:<30} RF = {rf:4d}px  {bar}')

# Translate to meters for Sentinel-2 (10m/pixel)
print()
print('In physical scale (Sentinel-2, 10m/pixel):')
for layer_name, rf in vgg_like[::3]:  # sample every 3
    coverage = rf * 10
    print(f'  After {layer_name:<28}: {rf}px = {coverage}m coverage')

In [ ]:
# Visualise receptive field growth
fig, ax = plt.subplots(figsize=(12, 5))

names = [h[0] for h in vgg_like]
rfs = [h[1] for h in vgg_like]

colors = ['#2196F3' if 'Conv' in n else '#F44336' for n in names]

ax.bar(range(len(rfs)), rfs, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Receptive Field (pixels)')
ax.set_title('Receptive Field Growth Through a VGG-like Network (3×3 conv, 2×MaxPool)\n'
             'Blue = Conv layer, Red = MaxPool layer', fontsize=11)

# Add ground coverage annotation
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim()[0] * 10, ax.get_ylim()[1] * 10)
ax2.set_ylabel('Ground Coverage at 10m/px (meters)', color='gray')
ax2.tick_params(axis='y', labelcolor='gray')

ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('MaxPool (red) layers have the biggest impact — they double the effective stride,'
      ' rapidly expanding the receptive field.')

## 2.5 — nn.Conv2d in PyTorch: All Parameters Explained

nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias, ...)

- in_channels:  number of input feature maps (3 for RGB, 64 after first layer)
- out_channels: number of FILTERS to learn (= number of output feature maps)
- kernel_size:  spatial size of each filter (scalar = square, tuple = rectangular)
- stride:       step size of the sliding window (default=1)
- padding:      zero-padding added to each side (default=0)
- bias:         learned per-channel bias term (default=True; set False with BN)

Weight tensor shape: (out_channels, in_channels, kH, kW)
Number of parameters: out_ch * in_ch * kH * kW + out_ch (bias)

In [ ]:
# Understanding parameter counts
configs = [
    ('First layer: RGB→32 ch, 3×3', nn.Conv2d(3, 32, 3, padding=1, bias=False)),
    ('First layer: RGB→32 ch, 5×5', nn.Conv2d(3, 32, 5, padding=2, bias=False)),
    ('First layer: RGB→32 ch, 7×7', nn.Conv2d(3, 32, 7, padding=3, bias=False)),
    ('Mid layer: 64→128 ch, 3×3', nn.Conv2d(64, 128, 3, padding=1, bias=False)),
    ('1×1 conv: 128→64 (bottleneck)', nn.Conv2d(128, 64, 1, bias=False)),
    ('Depthwise: 64→64 (groups=64)', nn.Conv2d(64, 64, 3, padding=1, groups=64, bias=False)),
]

print(f'{"Layer":<42} {"Params":>10} {"Weight shape":>20}')
print('-' * 75)
for name, layer in configs:
    params = sum(p.numel() for p in layer.parameters())
    shape = str(tuple(layer.weight.shape))
    print(f'{name:<42} {params:>10,} {shape:>20}')

print()
print('Key insight: 3×3 convolutions are parameter-efficient.')
print('Two 3×3 convs = same RF as one 5×5 but fewer parameters (2×9 < 25).')
print('Three 3×3 convs = same RF as one 7×7 but far fewer params (3×9=27 < 49).')
print('This is VGGNet\'s core insight (Simonyan & Zisserman, 2014).')

In [ ]:
# Visualise what a randomly initialised first layer 'looks for'
# (Before training — random noise)

torch.manual_seed(42)
conv = nn.Conv2d(3, 16, 3, padding=1, bias=False)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(16):
    ax = axes[i // 8, i % 8]
    # Each filter: shape (3, 3, 3) — show as RGB image
    kernel = conv.weight.data[i].permute(1, 2, 0).numpy()  # (3, 3, 3)
    kernel = (kernel - kernel.min()) / (kernel.max() - kernel.min() + 1e-8)
    ax.imshow(kernel)
    ax.set_title(f'Filter {i+1}', fontsize=8)
    ax.axis('off')

plt.suptitle('16 Randomly Initialised 3×3×3 CNN Filters (Before Training)\n'
             'After training, these will become edge/texture detectors', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Run the conv layer on real satellite data and visualise feature maps
# First, load an EuroSAT sample
sample = dataset[100]  # grab the 100th sample
img = sample['image'][:3].float() / 10000.0  # first 3 bands, normalize to 0-1

# Apply Sobel-X filter manually to one band
sobel_x_t = torch.tensor([[-1.,0.,1.],[-2.,0.,2.],[-1.,0.,1.]]).unsqueeze(0).unsqueeze(0)
sobel_y_t = torch.tensor([[-1.,-2.,-1.],[0.,0.,0.],[1.,2.,1.]]).unsqueeze(0).unsqueeze(0)

red_band = img[2:3].unsqueeze(0)  # (1, 1, 64, 64)

with torch.no_grad():
    feat_x = nn.functional.conv2d(red_band, sobel_x_t, padding=1).squeeze()
    feat_y = nn.functional.conv2d(red_band, sobel_y_t, padding=1).squeeze()
    feat_mag = torch.sqrt(feat_x**2 + feat_y**2)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(img.permute(1, 2, 0).numpy().clip(0, 1))
axes[0].set_title('Original RGB')

axes[1].imshow(feat_x.numpy(), cmap='RdBu_r')
axes[1].set_title('Sobel-X (vertical edges)')

axes[2].imshow(feat_y.numpy(), cmap='RdBu_r')
axes[2].set_title('Sobel-Y (horizontal edges)')

axes[3].imshow(feat_mag.numpy(), cmap='hot')
axes[3].set_title('Edge magnitude (√(X²+Y²))')

for ax in axes:
    ax.axis('off')

plt.suptitle(f'Edge Detection on EuroSAT Sample — Class: {dataset.classes[sample["label"].item()]}',
             fontsize=11)
plt.tight_layout()
plt.show()

## Practical Exercises

### Exercise 2.1 — Implement Strided Convolution
Modify `convolve2d_numpy` to support stride > 1.
Verify output size matches the formula: out = floor((H + 2*pad - k) / stride) + 1

### Exercise 2.2 — Parameter Counting
For each layer in SimpleCNN (Chapter 3), compute the number of:
  - learnable parameters (weights + biases)
  - multiply-accumulate operations (MACs) for a 64×64 input
Hint: MACs = out_ch × in_ch × kH × kW × out_H × out_W

### Exercise 2.3 — Kernel Visualisation
Train a SimpleCNN on EuroSAT for 2 epochs (Chapter 5 code).
Visualise the 32 learned filters from the first conv layer.
Compare with random init. Do you see Gabor-like patterns emerging?

### Mini-Project 2
Design 3 custom 3×3 kernels that detect:
  1. Diagonal edges (top-left to bottom-right)
  2. A specific texture pattern (e.g., regular grid structures)
  3. A colour-difference response (using multiple input channels)
Apply them to 5 different EuroSAT samples and show the results.

In [ ]:
# Exercise 2.1 Starter — Strided Convolution

def convolve2d_strided(image: np.ndarray, kernel: np.ndarray, 
                       stride: int = 1, padding: int = 0) -> np.ndarray:
    """
    TODO: Extend convolve2d_numpy to support stride > 1.
    
    Hint: output[i, j] = ... but i and j now step by `stride`
    """
    H, W = image.shape
    kH, kW = kernel.shape
    
    if padding > 0:
        image = np.pad(image, padding, mode='constant')
    
    H_pad, W_pad = image.shape
    out_H = (H_pad - kH) // stride + 1
    out_W = (W_pad - kW) // stride + 1
    
    output = np.zeros((out_H, out_W), dtype=np.float32)
    
    # TODO: Fill this loop (hint: use i*stride and j*stride as start indices)
    for i in range(out_H):
        for j in range(out_W):
            row_start = i * stride
            col_start = j * stride
            patch = image[row_start:row_start+kH, col_start:col_start+kW]
            output[i, j] = np.sum(patch * kernel)
    
    return output


# Verify
test = np.random.rand(8, 8).astype(np.float32)
k = np.ones((3, 3)) / 9  # average filter

for stride in [1, 2, 3]:
    result = convolve2d_strided(test, k, stride=stride, padding=1)
    # Verify with PyTorch
    t = torch.tensor(test).unsqueeze(0).unsqueeze(0)
    kt = torch.tensor(k.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    pt_result = nn.functional.conv2d(t, kt, stride=stride, padding=1).squeeze().numpy()
    match = np.allclose(result, pt_result, atol=1e-5)
    print(f'Stride={stride}: NumPy shape={result.shape}, PyTorch shape={pt_result.shape}, Match={match}')

print('\nChapter 2 complete! You now understand convolution at the mathematical level.')